In [0]:
%run /Workspace/Users/senoom222@gmail.com/databricks-code-repos-senthil/Databricks_workout_2025/Calling_1_wb_to_2_wb_using_util_run/Generic_Specific_Frame

In [0]:
dbutils.widgets.text("Catalog","")
CATALOG = dbutils.widgets.get("Catalog").strip()
dbutils.widgets.text("Schema","")
SCHEMA = dbutils.widgets.get("Schema").strip()

In [0]:
%python
import json

child_output = dbutils.notebook.run("/Workspace/Users/senoom222@gmail.com/databricks-code-repos-senthil/Databricks_workout_2025/Calling_1_wb_to_2_wb_using_util_run/config",120,{"Catalog":CATALOG,"Schema" : SCHEMA})

child_dict = json.loads(child_output)
CATALOG = child_dict["Catalog"]
SCHEMA = child_dict["Schema"]
SRC = child_dict["Source"]
BRONZE = child_dict["Bronze"]
SILVER = child_dict["Silver"]
GOLD = child_dict["Gold"]
SILVERDB = child_dict["Silver Table"]
GOLDDB = child_dict["Gold Table"]


print("Returned Source Location : ",SILVER)
print("Returned Target Location : ",GOLD)
print("Returned GOLDDB Location : ",GOLDDB)

In [0]:
from pyspark.sql.window import Window
staff = spark.read.format("delta").load(f"{SILVER}/logistics_silver_staff")
shipment = spark.read.format("delta").load(f"{SILVER}/logistics_silver_shipment")
joined = staff.join(shipment,how = "inner", on = "shipment_id")

curateddf = joined.select(
        "shipment_id",
    mask_name("staff_full_name").alias("masked_staff_name"),
    "role",
    "orgin_hub_city",
    "shipment_cost",
    "shipment_year",
    "shipment_month",
    "route_segment",
    "cost_per_kg",
    "tax_amount",
    "ingestion_timestamp"
)

deltawrite(
    curateddf,
    f"{GOLD}/logistics_gold_curated",
    "overwrite",
    "delta"
    )


GOLDDB = GOLDDB.split('/')[-1]
print(GOLDDB)

write_table
(
curateddf,
f"{GOLDDB}.gold_core_curated_table",
"overwrite",
"delta"
)
